# Homework: Inventory Allocation 

###  Bus 36109 "Advanced Decision Modeling with Python", Don Eisenstein
Don Eisenstein &copy; Copyright 2023, University of Chicago 

A retailer allocates units of a product at the start of each period to 6 retail stores.  Based on historical data, we have gathered 10 independent demand figures (scenarios) for a single period for each store.  These scenarios for each store are considered independent of each other and independent between one store and another.  Furthermore, each scenario is equally likely to occur.  

Each store also has its own per-unit penalties for stocking more units than demanded (Excess Penalty) or fewer units than demanded (Shortage Penalty).  Units left over one period are discarded (part of the Excess Penalty), and demand that exceeds supply is not back-ordered, but is lost (part of the Shortage Penalty).

The demand scenarios and excess and shortage penalties are given in the table below:

| Store | Excess Penalty per unit   | Shortage Penalty per unit | demand scenarios | 
| :--------: | ----------: | ------: | ----: | 
| A  | 10 | 10    | [100, 120, 100, 90, 80, 60, 70, 130, 110, 105] |
| B  | 20 | 15    | [80, 70, 50, 40, 50, 60, 90, 120, 85, 45] | 
| C  | 10 | 5 | [130, 150, 120, 160, 80, 170, 90, 85, 75, 105] | 
| D  | 5  | 5    | [25, 35, 50, 65, 45, 35, 70, 50, 40, 80] | 
| E  | 15  | 15    | [105, 95, 100, 90, 85, 110, 95, 100, 80, 100] | 
| F  | 15 | 10    | [100, 120, 100, 90, 80, 60, 70, 130, 110, 105] | 

The retailer seeks a policy to allocate exactly 500 units between the stores at the start of each period.  The policy will send the same store the same quantity every period.

Formulate and solve a 2-stage stochastic optimization model to determine how many integer units the retailer should send to each store at the start of each period to minimize the expected excess and shortage penalty costs.   

In [1]:
from pprint import pprint
import pulp

**Here is a var to store `total_units` and a list you can use called `stores`.** 

In [2]:
total_units = 500

stores = [
    {'name': 'A', 
     'excess_penalty': 10,
     'shortage_penalty': 10,
     'scenario_demands': [100, 120, 100, 90, 80, 60, 70, 130, 110, 105]
    }, 
    {'name': 'B', 
     'excess_penalty': 20,
     'shortage_penalty': 15,
     'scenario_demands': [80, 70, 50, 40, 50, 60, 90, 120, 85, 45]
    }, 
    {'name': 'C', 
     'excess_penalty': 10,
     'shortage_penalty': 5,
     'scenario_demands': [130, 150, 120, 160, 80, 170, 90, 85, 75, 105]
    }, 
    {'name': 'D', 
     'excess_penalty': 5,
     'shortage_penalty': 5,
     'scenario_demands': [25, 35, 50, 65, 45, 35, 70, 50, 40, 80]
    }, 
    {'name': 'E', 
     'excess_penalty': 15,
     'shortage_penalty': 15,
     'scenario_demands': [105, 95, 100, 90, 85, 110, 95, 100, 80, 100]
    }, 
    {'name': 'F', 
     'excess_penalty': 15,
     'shortage_penalty': 10,
     'scenario_demands': [100, 120, 100, 90, 80, 60, 70, 130, 110, 105]
    }
]

## Your Solution


**1. In broad terms, what are the variables, objective and constraints of this problem? You don't need to list the entire formulation. Just describe the structure/characteristics of your model.**

# The stage 1 variables are how many units to allocate to each store
# The stage 2 variables are the amount of excess (or shortage) for each store in each scenario.
# the objective is to minimize the over or under stocking of the stores. So in essence it is minimize the sum total of excess or shortage x the respective penalty cost x .1
# the constraint for stage 1 is that all all allocations to the 6 stores must equal 500 units. The stage 2 constraint requires that allocated store amount + shortage - excess must equal demand at a given store. 

**2. Create a PuLP LpProblem object and store it in the variable `model`.** 

In [3]:
model = pulp.LpProblem("Inventory_Allocation_Problem", pulp.LpMinimize)

**3. Create nicely named dictionaries to store your Stage 1 and Stage 2 variables.** 

In [4]:
# Stage 1 variables for allocation per store
store_names = [s['name'] for s in stores]
allocation_variables = pulp.LpVariable.dict("Allocation", store_names, lowBound = 0, cat='Integer')

# Stage 2 Variables to determine excess or shortage amount based on the scenario
excess_variables = {}
shortage_variables = {}
for store in stores:
    number_scenarios = len(store['scenario_demands'])

    for i in range(number_scenarios):
        key = (store['name'], i)
        
        excess_variables[key] = pulp.LpVariable(f"Excess_{store['name']}_{i}", lowBound = 0, cat = 'Integer')
        shortage_variables[key] = pulp.LpVariable(f"Shortage_{store['name']}_{i}", lowBound = 0, cat = 'Integer')


**4. Add your objective function to your `model`.**

In [6]:
prob = 1.0/10

objective_terms = []

for store in stores:
    name = store['name']
    excess_cost = store['excess_penalty']
    shortage_cost = store['shortage_penalty']

    for i in range(len(store['scenario_demands'])):
        excess_var = excess_variables[(name, i)]
        shortage_var = shortage_variables[(name, i)]

        scenario_cost = prob * ((excess_var * excess_cost) + (shortage_var * shortage_cost))

        objective_terms.append(scenario_cost)

model += pulp.lpSum(objective_terms)

**5. Add the constraints to your `model`.**

In [7]:
# Stage 1 constraint
model += pulp.lpSum(allocation_variables.values()) == total_units, "Total_Allocation_Constraint"

#Stage 2 Constraints
for store in stores:
    name = store['name']
    demands = store['scenario_demands']

    for i, demand_value in enumerate(demands):
        allocated = allocation_variables[name]
        excess = excess_variables[(name, i)]
        shortage = shortage_variables[(name, i)]

        constraint_name = f"Balance_{name}_Scenario_{i}"
        
        model += (allocated + shortage - excess == demand_value), constraint_name
        

**6. Print your `model` and verify that the variables, objective and constraints all look correct.**

In [11]:
print(model)

Inventory_Allocation_Problem:
MINIMIZE
1.0*Excess_A_0 + 1.0*Excess_A_1 + 1.0*Excess_A_2 + 1.0*Excess_A_3 + 1.0*Excess_A_4 + 1.0*Excess_A_5 + 1.0*Excess_A_6 + 1.0*Excess_A_7 + 1.0*Excess_A_8 + 1.0*Excess_A_9 + 2.0*Excess_B_0 + 2.0*Excess_B_1 + 2.0*Excess_B_2 + 2.0*Excess_B_3 + 2.0*Excess_B_4 + 2.0*Excess_B_5 + 2.0*Excess_B_6 + 2.0*Excess_B_7 + 2.0*Excess_B_8 + 2.0*Excess_B_9 + 1.0*Excess_C_0 + 1.0*Excess_C_1 + 1.0*Excess_C_2 + 1.0*Excess_C_3 + 1.0*Excess_C_4 + 1.0*Excess_C_5 + 1.0*Excess_C_6 + 1.0*Excess_C_7 + 1.0*Excess_C_8 + 1.0*Excess_C_9 + 0.5*Excess_D_0 + 0.5*Excess_D_1 + 0.5*Excess_D_2 + 0.5*Excess_D_3 + 0.5*Excess_D_4 + 0.5*Excess_D_5 + 0.5*Excess_D_6 + 0.5*Excess_D_7 + 0.5*Excess_D_8 + 0.5*Excess_D_9 + 1.5*Excess_E_0 + 1.5*Excess_E_1 + 1.5*Excess_E_2 + 1.5*Excess_E_3 + 1.5*Excess_E_4 + 1.5*Excess_E_5 + 1.5*Excess_E_6 + 1.5*Excess_E_7 + 1.5*Excess_E_8 + 1.5*Excess_E_9 + 1.5*Excess_F_0 + 1.5*Excess_F_1 + 1.5*Excess_F_2 + 1.5*Excess_F_3 + 1.5*Excess_F_4 + 1.5*Excess_F_5 + 1.5*Exces

**7. Solve your `model` and print the optimal objective value.**

In [12]:
model.solve()

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /opt/conda/lib/python3.13/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/5730baec5e56441b850717b01661ea5a-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/5730baec5e56441b850717b01661ea5a-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 66 COLUMNS
At line 625 RHS
At line 687 BOUNDS
At line 814 ENDATA
Problem MODEL has 61 rows, 126 columns and 186 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1057.5 - 0.00 seconds
Cgl0003I 0 fixed, 120 tightened bounds, 0 strengthened rows, 0 substitutions
Cgl0004I processed model has 61 rows, 126 columns (126 integer (0 of which binary)) and 186 elements
Cutoff increment increased from 1e-05 to 0.4999
Cbc0012I Integer solution of 1057.5 found by DiveCoefficient after 0 iterations and 0 nodes (0.00 seconds)
Cbc0001I Search

1

**8. Print out the optimal number of units allocated to each store.**

In [13]:
for store_name in store_names:
    variable = allocation_variables[store_name]
    amount = variable.varValue

    print(f"Store {store_name}: {amount} units")

Store A: 100.0 units
Store B: 60.0 units
Store C: 90.0 units
Store D: 50.0 units
Store E: 100.0 units
Store F: 100.0 units
